In [14]:
import psycopg2 as pg

conexao = None

try:
    conexao = pg.connect(
        host='localhost',
        user='postgres',
        password='postgres',
        port='5432',
        database='lab365'
    )

    with conexao.cursor() as cursor:
        print("Conexão bem sucedida! Buscando dados...")

        cursor.execute("SELECT * FROM cliente;")
        registros = cursor.fetchall()

        print(f"Encontrados {len(registros)} registros:")
        for linha in registros:
            print(linha)


except Exception as erro:
    print("Ocorreu um erro na conexão:")
    print(erro)

finally:
    if conexao:
        conexao.close()
        print("Conexão encerrada com segurança.")

Conexão bem sucedida! Buscando dados...
Encontrados 3 registros:
(1, 'Marcos Enrico')
(2, 'Davi da Silva')
(3, 'Rosane Silva')
Conexão encerrada com segurança.


In [1]:
import psycopg2 as pg

def connect_db():
    try:
        return pg.connect(
            host='localhost',
            user='postgres',
            password='postgres',
            port='5432',
            database='lab365'
        )
    except:
        print("Erro: Falha ao conectar com o banco de dados!")

In [4]:
conn = connect_db()

with conn.cursor() as cursor:
    print("Conexão bem sucedida! Buscando dados...")

    cursor.execute("SELECT * FROM cliente;")
    registros = cursor.fetchall()

    print(f"Encontrados {len(registros)} registros:")
    for linha in registros:
        print(linha)


conn.close()

Conexão bem sucedida! Buscando dados...
Encontrados 3 registros:
(1, 'Marcos Enrico')
(2, 'Davi da Silva')
(3, 'Rosane Silva')


In [13]:
def cadastrar_cliente(nome: str):
    conn = None
    try:
        conn = connect_db()
        if conn is None:
            print("Não foi possível conectar ao banco.")
            return

        with conn.cursor() as cursor:
            cursor.execute("INSERT INTO cliente(nome_cliente) VALUES (%s);", (nome,))
            conn.commit()
            print("Cliente cadastrado com sucesso.")

    except Exception as e:
        if conn:
            conn.rollback()
        print("Erro: Falha ao cadastrar cliente:", e)

    finally:
        if conn:
            conn.close()

cadastrar_cliente("Carlos")

Cliente cadastrado com sucesso.


In [ ]:
def listar_alunos_por_turma(turma_id: id):
    try:
        with connect_db() as conn:
            with conn.cursor() as cursor:

                cursor.execute("""
                    SELECT a.nome,
                    FROM tbl_aluno a
                    JOIN tbl_aluno_has_turma at ON a.id_aluno = at.fk_aluno
                    WHERE at.fk_turma = %s;
                """, (turma_id,))

                registros = cursor.fetchall()

                print(f"Encontrados {len(registros)} registros:")

                for linha in registros:
                    print(linha)

    except Exception as e:
        print("Erro: Falha ao listar alunos por turma:", e)

In [16]:
# python
import psycopg2 as pg
from contextlib import contextmanager

def connect_db(host='localhost', user='postgres', password='postgres', port='5432', database='lab365'):
    try:
        return pg.connect(host=host, user=user, password=password, port=port, database=database)
    except Exception as e:
        print("Erro ao conectar:", e)
        return None

@contextmanager
def db_cursor(**conn_kwargs):
    """
    Context manager que retorna (conn, cursor).
    - Faz commit quando o bloco termina sem exceção.
    - Faz rollback se ocorrer exceção.
    - Fecha a conexão sempre.
    """
    conn = connect_db(**conn_kwargs)
    if conn is None:
        raise RuntimeError("Não foi possível abrir a conexão com o banco.")
    try:
        with conn.cursor() as cursor:
            try:
                yield conn, cursor
                conn.commit()
            except Exception:
                conn.rollback()
                raise
    finally:
        conn.close()

# Exemplo de funções CRUD usando o contexto acima:

def criar_cliente(nome: str):
    try:
        with db_cursor() as (_, cursor):
            cursor.execute("INSERT INTO cliente(nome_cliente) VALUES (%s);", (nome,))
            # commit é feito automaticamente pelo context manager
    except Exception as e:
        print("Erro ao criar cliente:", e)

def ler_clientes():
    try:
        with db_cursor() as (_, cursor):
            cursor.execute("SELECT cliente_id, nome_cliente FROM cliente;")
            return cursor.fetchall()
    except Exception as e:
        print("Erro ao ler clientes:", e)
        return []

def atualizar_cliente(cliente_id: int, novo_nome: str):
    try:
        with db_cursor() as (_, cursor):
            cursor.execute("UPDATE cliente SET nome_cliente = %s WHERE id = %s;", (novo_nome, cliente_id))
    except Exception as e:
        print("Erro ao atualizar cliente:", e)

def deletar_cliente(cliente_id: int):
    try:
        with db_cursor() as (_, cursor):
            cursor.execute("DELETE FROM cliente WHERE id = %s;", (cliente_id,))
    except Exception as e:
        print("Erro ao deletar cliente:", e)

In [17]:
ler_clientes()

[(1, 'Marcos Enrico'),
 (2, 'Davi da Silva'),
 (3, 'Rosane Silva'),
 (4, 'Pedro'),
 (5, 'Carlos')]